In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams['animation.embed_limit'] = 200.0

# --- 1. Physical & Numerical Parameters ---
N = 150               # Lattice size (150x150)
dt = 0.04             # Time step for numerical stability
steps_per_frame = 30
J = 1.0               # Spatial coupling constant (discrete Laplacian)
lam = 2.0             # Spontaneous U(1) symmetry-breaking scale
eta = 1.0             # Vacuum expectation value (VEV)
m_param = 0.15        # Explicit symmetry-breaking parameter (wall thickness/tension)

# Field arrays: phi = x + i*y
x = np.zeros((N, N))
y = np.zeros((N, N))
vx = np.zeros((N, N))
vy = np.zeros((N, N))

# Vortex and antivortex core coordinates
v1_cx, v1_cy = N // 2 - 25, N // 2  # Vortex (Q = +1)
v2_cx, v2_cy = N // 2 + 25, N // 2  # Antivortex (Q = -1)

# --- 2. Field Initialization (Vortex-Antivortex Pair) ---
for i in range(N):
    for j in range(N):
        dx1, dy1 = j - v1_cx, i - v1_cy
        r1 = np.sqrt(dx1**2 + dy1**2)
        theta1 = np.arctan2(dy1, dx1)

        dx2, dy2 = j - v2_cx, i - v2_cy
        r2 = np.sqrt(dx2**2 + dy2**2)
        theta2 = np.arctan2(dy2, dx2)

        # Total topological phase (Q1 + Q2 = +1 - 1 = 0)
        theta_total = theta1 - theta2

        # Product ansatz for amplitude (vanishes at both cores)
        amp = eta * np.tanh(r1 / 4.0) * np.tanh(r2 / 4.0)

        x[i, j] = amp * np.cos(theta_total)
        y[i, j] = amp * np.sin(theta_total)


# --- 3. Equations of Motion ---
def get_accelerations(x_pos, y_pos):
    """Compute field accelerations from the 2D lattice Lagrangian with periodic BCs."""
    lap_x = (
        np.roll(x_pos, 1, axis=0) + np.roll(x_pos, -1, axis=0)
        + np.roll(x_pos, 1, axis=1) + np.roll(x_pos, -1, axis=1) - 4 * x_pos
    )
    lap_y = (
        np.roll(y_pos, 1, axis=0) + np.roll(y_pos, -1, axis=0)
        + np.roll(y_pos, 1, axis=1) + np.roll(y_pos, -1, axis=1) - 4 * y_pos
    )

    phi_sq = x_pos**2 + y_pos**2

    # U(1)-symmetric Higgs potential forces
    f_higgs_x = -lam**2 * x_pos * (phi_sq - eta**2)
    f_higgs_y = -lam**2 * y_pos * (phi_sq - eta**2)

    # Explicit U(1) symmetry-breaking force from -m^3 / (2*eta) * (phi + phi*)
    f_explicit_x = (m_param**3) / eta
    f_explicit_y = 0.0

    ax = J * lap_x + f_higgs_x + f_explicit_x
    ay = J * lap_y + f_higgs_y + f_explicit_y

    return ax, ay


# --- 4. Dual-Panel Visualization Setup ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

phase = np.arctan2(y, x)
im = ax1.imshow(phase, cmap='twilight', origin='lower', extent=[0, N, 0, N], vmin=-np.pi, vmax=np.pi)
ax1.set_title("String-Wall System Phase (Domain Wall)")
ax1.set_xlabel("X Index")
ax1.set_ylabel("Y Index")
fig.colorbar(im, ax=ax1, label="Phase (radians)")

skip = 4
X, Y = np.meshgrid(np.arange(0, N, skip), np.arange(0, N, skip))
Q = ax2.quiver(X, Y, x[::skip, ::skip], y[::skip, ::skip], color='darkblue', scale=20, pivot='mid')
ax2.set_xlim(0, N)
ax2.set_ylim(0, N)
ax2.set_aspect('equal')
ax2.set_title("Vector Field (Re(phi), Im(phi))")
ax2.set_xlabel("X Index")
ax2.set_ylabel("Y Index")
ax2.grid(True, alpha=0.2)


# --- 5. Symplectic Euler Time Evolution ---
def update(frame):
    global x, y, vx, vy
    for _ in range(steps_per_frame):
        ax, ay = get_accelerations(x, y)
        vx += ax * dt
        vy += ay * dt
        x += vx * dt
        y += vy * dt

    if frame >= 50 and frame % 50 == 0:
        print(f"Frame: {frame}")

    new_phase = np.arctan2(y, x)
    im.set_array(new_phase)
    Q.set_UVC(x[::skip, ::skip], y[::skip, ::skip])

    return [im, Q]


ani = FuncAnimation(fig, update, frames=200, interval=50, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Parameters ---
N = 150
eta = 1.0

# Vortex and antivortex coordinates (separated by L = 50 lattice units)
v1_cx, v1_cy = N // 2 - 25, N // 2  # Vortex (Q = +1)
v2_cx, v2_cy = N // 2 + 25, N // 2  # Antivortex (Q = -1)

# --- 2. Vectorized Initialization ---
X, Y = np.meshgrid(np.arange(N), np.arange(N))

dx1, dy1 = X - v1_cx, Y - v1_cy
dx2, dy2 = X - v2_cx, Y - v2_cy

r1 = np.sqrt(dx1**2 + dy1**2)
r2 = np.sqrt(dx2**2 + dy2**2)

theta1 = np.arctan2(dy1, dx1)
theta2 = np.arctan2(dy2, dx2)

# Superposition ansatz: product of amplitudes, difference of phases
amplitude = eta * np.tanh(r1 / 4.0) * np.tanh(r2 / 4.0)
theta_total = theta1 - theta2

field_x = amplitude * np.cos(theta_total)
field_y = amplitude * np.sin(theta_total)

# Regularize exact core singularities
field_x[r1 < 1e-5] = 0.0
field_y[r1 < 1e-5] = 0.0
field_x[r2 < 1e-5] = 0.0
field_y[r2 < 1e-5] = 0.0

# --- 3. Visualization ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

im = ax1.imshow(theta_total, cmap='twilight', origin='lower')
ax1.scatter([v1_cx, v2_cx], [v1_cy, v2_cy], color='white', marker='o', edgecolors='black', s=100, label='Cores')
ax1.set_title("Field Phase: Domain Wall (Kink) Between Vortex & Antivortex")
ax1.set_xlabel("X Index")
ax1.set_ylabel("Y Index")
fig.colorbar(im, ax=ax1, label="Phase (radians)")
ax1.legend()

ax2.plot(X[N // 2, :], theta_total[N // 2, :], color='darkred', lw=2)
ax2.axvline(v1_cx, color='gray', linestyle='--', label='Vortex (+1)')
ax2.axvline(v2_cx, color='gray', linestyle=':', label='Antivortex (-1)')
ax2.set_title("Phase Kink Profile Along Central Axis (Y = N // 2)")
ax2.set_xlabel("X Index")
ax2.set_ylabel("Phase (radians)")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Parameters ---
N = 150
eta = 1.0

v1_cx, v1_cy = N // 2 - 25, N // 2       # Vortex 1 (Q = +1)
v2_cx, v2_cy = N // 2 + 25, N // 2       # Antivortex (Q = -1)
v3_cx, v3_cy = N // 2, N // 2 + 35       # Unpaired Vortex (Q = +1)

# --- 2. Vectorized Lattice Initialization ---
X, Y = np.meshgrid(np.arange(N), np.arange(N))

dx1, dy1 = X - v1_cx, Y - v1_cy
dx2, dy2 = X - v2_cx, Y - v2_cy
dx3, dy3 = X - v3_cx, Y - v3_cy

r1, theta1 = np.sqrt(dx1**2 + dy1**2), np.arctan2(dy1, dx1)
r2, theta2 = np.sqrt(dx2**2 + dy2**2), np.arctan2(dy2, dx2)
r3, theta3 = np.sqrt(dx3**2 + dy3**2), np.arctan2(dy3, dx3)

# Total topological phase (net global charge Q = +1 - 1 + 1 = +1)
theta_total = theta1 - theta2 + theta3

# Wrap phase into [-pi, pi]
theta_total = np.remainder(theta_total + np.pi, 2 * np.pi) - np.pi

amplitude = eta * np.tanh(r1 / 4.0) * np.tanh(r2 / 4.0) * np.tanh(r3 / 4.0)

field_x = amplitude * np.cos(theta_total)
field_y = amplitude * np.sin(theta_total)

for r in (r1, r2, r3):
    field_x[r < 1e-5] = 0.0
    field_y[r < 1e-5] = 0.0

# --- 3. Visualization ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

im = ax1.imshow(theta_total, cmap='twilight', origin='lower')
ax1.scatter([v1_cx, v2_cx, v3_cx], [v1_cy, v2_cy, v3_cy], color='white', marker='o', edgecolors='black', s=100)
ax1.text(v1_cx, v1_cy + 5, "+1", color='white', weight='bold')
ax1.text(v2_cx, v2_cy + 5, "-1", color='white', weight='bold')
ax1.text(v3_cx, v3_cy + 5, "Unpaired +1", color='white', weight='bold')
ax1.set_title("Three-Vortex Phase Map (Global Charge Q = +1)")
ax1.set_xlabel("X Index")
ax1.set_ylabel("Y Index")

skip = 5
ax2.quiver(X[::skip, ::skip], Y[::skip, ::skip], field_x[::skip, ::skip], field_y[::skip, ::skip], color='darkblue', scale=22)
ax2.set_title("Vector Field on 2D Torus")
ax2.set_xlabel("X Index")
ax2.set_ylabel("Y Index")

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams['animation.embed_limit'] = 200.0

# --- 1. Parameters ---
N = 150
dt = 0.04
steps_per_frame = 30
J = 1.0
lam = 2.0
eta = 1.0
m_param = 0.15

vx = np.zeros((N, N))
vy = np.zeros((N, N))

v1_cx, v1_cy = N // 2 - 25, N // 2
v2_cx, v2_cy = N // 2 + 25, N // 2
v3_cx, v3_cy = N // 2, N // 2 + 35

# --- 2. Vectorized Three-Vortex Initialization ---
X_grid, Y_grid = np.meshgrid(np.arange(N), np.arange(N))

dx1, dy1 = X_grid - v1_cx, Y_grid - v1_cy
dx2, dy2 = X_grid - v2_cx, Y_grid - v2_cy
dx3, dy3 = X_grid - v3_cx, Y_grid - v3_cy

r1, theta1 = np.sqrt(dx1**2 + dy1**2), np.arctan2(dy1, dx1)
r2, theta2 = np.sqrt(dx2**2 + dy2**2), np.arctan2(dy2, dx2)
r3, theta3 = np.sqrt(dx3**2 + dy3**2), np.arctan2(dy3, dx3)

theta_total = np.remainder((theta1 - theta2 + theta3) + np.pi, 2 * np.pi) - np.pi
amp = eta * np.tanh(r1 / 4.0) * np.tanh(r2 / 4.0) * np.tanh(r3 / 4.0)

x = amp * np.cos(theta_total)
y = amp * np.sin(theta_total)

for r in (r1, r2, r3):
    x[r < 1e-5] = 0.0
    y[r < 1e-5] = 0.0


# --- 3. Equations of Motion ---
def get_accelerations(x_pos, y_pos):
    lap_x = (
        np.roll(x_pos, 1, axis=0) + np.roll(x_pos, -1, axis=0)
        + np.roll(x_pos, 1, axis=1) + np.roll(x_pos, -1, axis=1) - 4 * x_pos
    )
    lap_y = (
        np.roll(y_pos, 1, axis=0) + np.roll(y_pos, -1, axis=0)
        + np.roll(y_pos, 1, axis=1) + np.roll(y_pos, -1, axis=1) - 4 * y_pos
    )

    phi_sq = x_pos**2 + y_pos**2
    f_higgs_x = -lam**2 * x_pos * (phi_sq - eta**2)
    f_higgs_y = -lam**2 * y_pos * (phi_sq - eta**2)

    f_explicit_x = (m_param**3) / eta
    f_explicit_y = 0.0

    return J * lap_x + f_higgs_x + f_explicit_x, J * lap_y + f_higgs_y + f_explicit_y


# --- 4. Visualization Setup ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

phase = np.arctan2(y, x)
im = ax1.imshow(phase, cmap='twilight', origin='lower', extent=[0, N, 0, N], vmin=-np.pi, vmax=np.pi)
ax1.set_title("Three-Vortex Phase Evolution on 2D Torus (Domain Walls)")
ax1.set_xlabel("X Index")
ax1.set_ylabel("Y Index")
fig.colorbar(im, ax=ax1, label="Phase (radians)")

skip = 4
X, Y = np.meshgrid(np.arange(0, N, skip), np.arange(0, N, skip))
Q = ax2.quiver(X, Y, x[::skip, ::skip], y[::skip, ::skip], color='darkblue', scale=20, pivot='mid')
ax2.set_xlim(0, N)
ax2.set_ylim(0, N)
ax2.set_aspect('equal')
ax2.set_title("Vector Field (Re(phi), Im(phi))")
ax2.set_xlabel("X Index")
ax2.set_ylabel("Y Index")
ax2.grid(True, alpha=0.2)


# --- 5. Time Evolution Loop ---
def update(frame):
    global x, y, vx, vy
    for _ in range(steps_per_frame):
        ax, ay = get_accelerations(x, y)
        vx += ax * dt
        vy += ay * dt
        x += vx * dt
        y += vy * dt

    if frame >= 50 and frame % 50 == 0:
        print(f"Frame: {frame}")

    new_phase = np.arctan2(y, x)
    im.set_array(new_phase)
    Q.set_UVC(x[::skip, ::skip], y[::skip, ::skip])

    return [im, Q]


#ani = FuncAnimation(fig, update, frames=200, interval=50, blit=True)
#plt.close(fig)
#HTML(ani.to_jshtml())

from matplotlib.animation import PillowWriter

ani = FuncAnimation(fig, update, frames=180, interval=30, blit=False)
plt.close(fig)

# dpi=70 და fps=15 ზომას 3-5 MB-მდე ჩამოიყვანს, რასაც GitHub უპრობლემოდ აჩვენებს
ani.save("soliton_animation.gif", writer=PillowWriter(fps=15), dpi=70)